In [1]:
import pandas as pd
from pathlib import Path

import matplotlib
from matplotlib import pyplot as plt

In [2]:
current_path = Path.cwd()
file_path = current_path.parent / 'data' / 'all nodes.csv'
df = pd.read_csv(file_path)
df

,hostid,Хост,itemid,Элемент,Ключ,Тип данных,Числовое значение,Время числового значения
0,10084,Zabbix server,42269,Linux: CPU utilization,system.cpu.util,0,34.509983,1760503674
1,10084,Zabbix server,42269,Linux: CPU utilization,system.cpu.util,0,34.778345,1760503704
2,10084,Zabbix server,42269,Linux: CPU utilization,system.cpu.util,0,33.764368,1760503734
3,10084,Zabbix server,42269,Linux: CPU utilization,system.cpu.util,0,33.232588,1760503764
4,10084,Zabbix server,42269,Linux: CPU utilization,system.cpu.util,0,36.245971,1760503794
...,...,...,...,...,...,...,...,...
6974599,11565,node480,78856,Linux: CPU utilization,system.cpu.util,0,0.047240,1761118851
6974600,11565,node480,78856,Linux: CPU utilization,system.cpu.util,0,0.043767,1761118881
6974601,11565,node480,78856,Linux: CPU utilization,system.cpu.util,0,0.043766,1761118911
6974602,11565,node480,78856,Linux: CPU utilization,system.cpu.util,0,0.043766,1761118941


In [ ]:
# шаг 1: разделить данные для каждого узла отдельно

In [ ]:
# шаг 2: для каждого узла собрать обработанный файл вида: timestamp,cpu,voltage,cpu_z,voltage_z,relation_signal

Вариант B. Общая схема, но отдельная адаптация порогов по узлам

Это обычно лучше на практике:

модель одна;
но порог аномальности и/или базовая статистика — отдельные для каждого узла.

То есть сам детектор общий, а чувствительность подстраивается на уровне узла.

Для вашей задачи я бы рекомендовал именно этот путь:

сначала одна модель на node001 как baseline;
потом проверка на группе других узлов;
потом либо одна общая модель, либо общая модель + per-node threshold.

Как организовать realtime-имитацию

Если вы хотите имитировать поток данных “как будто сейчас”, но ускоренно, то нужен replay-микросервис.

Он работает так:

читает подготовленные файлы узлов;
у каждого узла держит текущую позицию;
через заданный ускоренный интервал подает следующую точку;
собирает окно последних m точек;
прогоняет окно через модель;
публикует метрики в Prometheus.

То есть офлайн-файлы становятся источником “псевдо-стриминга”.

Какой должна быть структура микросервиса

Лучше разделить его логически на 4 части.

1. Loader / Replay Engine

Отвечает за чтение подготовленных файлов по узлам:

node001.csv
node002.csv
…

Он знает:

текущий индекс по каждому узлу;
скорость воспроизведения;
когда отдавать следующую точку.
2. State Manager

Для каждого узла хранит:

буфер последних точек;
текущее окно;
последнее предсказание;
статус аномалии;
время последнего обновления.

То есть у каждого узла свой “контекст”.

3. Inference Engine

Берет буфер узла и:

формирует окно длины m;
нормализует его так же, как на обучении;
прогоняет через модель;
получает score;
сравнивает с threshold;
выдает is_anomaly.
4. Prometheus Exporter

Публикует результаты как метрики:

исходные replay-метрики;
score модели;
бинарный флаг аномалии.